<a href="https://colab.research.google.com/github/ZainAfzalHashmi/Assignments-3/blob/main/Password_strenght.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:

# --- Password Strength Logic ---

def assess_password_strength(password):
    score = 0
    feedback = []

    # 1. Length
    length = len(password)
    if length >= 12:
        score += 3
        feedback.append("Excellent length!")
    elif length >= 8:
        score += 2
        feedback.append("Good length.")
    else:
        feedback.append("Password is too short. Aim for at least 8 characters, preferably 12+.")

    # 2. Character Types
    has_lowercase = bool(re.search(r'[a-z]', password))
    has_uppercase = bool(re.search(r'[A-Z]', password))
    has_digit = bool(re.search(r'\d', password))
    has_symbol = bool(re.search(r'[!@#$%^&*()_+={}\[\]:;"<>,.?/~`\-]', password))

    char_types = sum([has_lowercase, has_uppercase, has_digit, has_symbol])

    if char_types == 4:
        score += 4
        feedback.append("Contains a mix of uppercase, lowercase, numbers, and symbols.")
    elif char_types == 3:
        score += 3
        feedback.append("Contains a good mix of character types.")
    elif char_types == 2:
        score += 1
        feedback.append("Consider adding more character types (e.g., symbols, numbers).")
    else:
        feedback.append("Add a mix of character types (uppercase, lowercase, numbers, symbols).")

    # 3. Avoid Common Patterns
    # Sequential numbers
    if re.search(r'123|234|345|456|567|678|789|012', password) or \
       re.search(r'321|432|543|654|765|876|987|210', password):
        score -= 2
        feedback.append("Avoid sequential numbers (e.g., 123, 321).")
    # Sequential letters
    if re.search(r'abc|bcd|cde|def|efg|fgh|ghi|hij|ijk|jkl|klm|lmn|mno|nop|opq|pqr|qrs|rst|stu|tuv|uvw|vwx|wxy|xyz', password, re.IGNORECASE) or \
       re.search(r'zyx|yxw|xwv|wvu|vut|uts|tsr|srq|rqp|qpo|pon|onm|nml|mlk|lkj|kji|jih|ihg|hgf|gfe|fed|edc|dcb|cba', password, re.IGNORECASE):
        score -= 2
        feedback.append("Avoid sequential letters (e.g., abc, cba).")
    # Repeated characters
    if re.search(r'(.)\1\1', password): # Detects three or more of the same character
        score -= 1
        feedback.append("Avoid repeating characters (e.g., 'aaa').")

    # 4. Entropy Calculation (Approximation)
    # Estimate character set size
    charset_size = 0
    if has_lowercase: charset_size += 26
    if has_uppercase: charset_size += 26
    if has_digit: charset_size += 10
    if has_symbol: charset_size += 32 # Common symbols

    entropy = 0
    if charset_size > 0:
        entropy = length * math.log2(charset_size)

    # 5. Zxcvbn Integration (for more sophisticated analysis)
    # This library is very good at pattern matching and common password detection
    zxcvbn_result = zxcvbn(password)
    zxcvbn_score = zxcvbn_result['score'] # 0-4
    zxcvbn_feedback = zxcvbn_result['feedback']['suggestions']

    # Adjust overall score based on zxcvbn (optional, can just display its score)
    # For simplicity, we'll map zxcvbn_score directly to a strength level
    # A more complex integration might add/subtract points based on its strength.

    # Determine overall strength level
    if zxcvbn_score == 4:
        strength_level = "Excellent"
        color = "green"
    elif zxcvbn_score == 3:
        strength_level = "Good"
        color = "lightgreen"
    elif zxcvbn_score == 2:
        strength_level = "Fair"
        color = "orange"
    elif zxcvbn_score == 1:
        strength_level = "Weak"
        color = "red"
    else: # zxcvbn_score == 0
        strength_level = "Very Weak"
        color = "darkred"

    # Combine feedback
    if zxcvbn_feedback:
        for item in zxcvbn_feedback:
            if item not in feedback: # Avoid duplicating general feedback
                feedback.append(item)
    elif zxcvbn_result['feedback']['warning']:
        feedback.append(zxcvbn_result['feedback']['warning'])


    return {
        "score": score, # Our custom score (can be refined or removed)
        "strength_level": strength_level,
        "color": color,
        "feedback": feedback,
        "entropy": f"{entropy:.2f} bits",
        "zxcvbn_score": zxcvbn_score,
        "zxcvbn_feedback": zxcvbn_result['feedback']
    }